## legnet

In [1]:
import legnet
import torch
import hybrid

LEGNET_MODEL_PATH = '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/oracle_ckpt/LegNet_80.ckpt' #pretraned legNet-predictor

device = 'cuda:0'
LegNet = legnet.SeqNN(150,
                ks=7,
                use_single_channel=True,
                block_sizes=[256, 128, 128, 64, 64, 64, 64],
                final_ch=18).to(device)

LegNet.load_state_dict(torch.load(LEGNET_MODEL_PATH, map_location=device))

UAS_MODEL_PATH = '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/oracle_ckpt/uas_hybridnet_oracle.ckpt' #pretraned legNet-predictor
UASNet = hybrid.HybridNet().to(device)

UAS_ckpt = torch.load(UAS_MODEL_PATH, map_location=device)

state_dict = {k.replace('model.', ''): v for k, v in UAS_ckpt['state_dict'].items()}
UASNet.load_state_dict(state_dict)


/tmp/ipykernel_786244/2267784863.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  LegNet.load_state_dict(torch.load(LEGNET_MODEL_PATH, map_location=device))
/tmp/ipykern

<All keys matched successfully>

In [2]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

# =========================
# One-hot encoders
# =========================
def str_to_one_hot_core(seq: str) -> torch.Tensor:
    """DNA -> one-hot for Core model (6 channels). Returns [L, 6]."""
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq = seq.upper()
    seq_tensor = torch.tensor([base_map[b] for b in seq], dtype=torch.long)
    seq_one_hot = torch.nn.functional.one_hot(seq_tensor, num_classes=4).float()

    singleton_channel = torch.ones((seq_one_hot.size(0), 1))   # singleton == 1
    reverse_channel = torch.zeros((seq_one_hot.size(0), 1))
    seq_one_hot = torch.cat([seq_one_hot, singleton_channel, reverse_channel], dim=1)  # [L,6]
    return seq_one_hot

def str_to_one_hot_uas(seq: str) -> torch.Tensor:
    """DNA -> one-hot for UAS model (4 channels). Returns [L, 4]."""
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq = seq.upper()
    seq_tensor = torch.tensor([base_map[b] for b in seq], dtype=torch.long)
    seq_one_hot = torch.nn.functional.one_hot(seq_tensor, num_classes=4).float()  # [L,4]
    return seq_one_hot

# =========================
# Core processor (window=80, step=20)
# =========================
class CoreProcessor:
    def __init__(self, model, device, prom_len=500, win=80, step=20):
        self.model = model
        self.device = device
        self.prom_len = prom_len
        self.win = win
        self.step = step
        # NOTE: end-exclusive, enforce non-overflow windows
        self.regions = {f"cp_{i}_{i+win}": (i, i+win)
                        for i in range(0, prom_len - win + 1, step)}  # 0..420

    def predict_single(self, seq: str) -> float:
        if not seq or len(seq) == 0:
            return 0.0

        # [1, L, 6] -> [1, 6, L]
        x = str_to_one_hot_core(seq).unsqueeze(0).to(self.device).permute(0, 2, 1)

        with torch.no_grad():
            self.model.eval()
            pred = self.model(x)
            if isinstance(pred, tuple):
                pred = pred[1]  # keep your original convention
        return float(pred.item())

    def process_batch(self, sequences):
        preds = []
        for seq in tqdm(sequences, desc="Predicting Core (80bp windows)"):
            preds.append(self.predict_single(seq))
        return preds

    def extract_regions(self, data: pd.DataFrame, seq_col="seq"):
        for name, (start, end) in self.regions.items():
            data[name] = data[seq_col].apply(lambda s: s[start:end] if isinstance(s, str) and len(s) >= end else "")

    def predict_regions(self, data: pd.DataFrame):
        for name in self.regions.keys():
            print(f"Predicting {name} ...")
            seqs = data[name].tolist()
            data[name] = self.process_batch(seqs)

# =========================
# UAS processor (window=334, step=20)
# =========================
class UASProcessor:
    def __init__(self, model, device, prom_len=500, win=334, step=20):
        self.model = model
        self.device = device
        self.prom_len = prom_len
        self.win = win
        self.step = step
        # NOTE: end-exclusive, enforce non-overflow windows
        self.regions = {f"uas_{i}_{i+win}": (i, i+win)
                        for i in range(0, prom_len - win + 1, step)}  # 0..166

    def predict_single(self, seq: str) -> float:
        if not seq or len(seq) == 0:
            return 0.0

        # [1, L, 4]
        x = str_to_one_hot_uas(seq).unsqueeze(0).to(self.device)

        with torch.no_grad():
            self.model.eval()
            pred = self.model(x)
            if isinstance(pred, tuple):
                pred = pred[0]  # keep your original convention
        return float(pred.item())

    def process_batch(self, sequences):
        preds = []
        for seq in tqdm(sequences, desc="Predicting UAS (334bp windows)"):
            preds.append(self.predict_single(seq))
        return preds

    def extract_regions(self, data: pd.DataFrame, seq_col="seq"):
        for name, (start, end) in self.regions.items():
            data[name] = data[seq_col].apply(lambda s: s[start:end] if isinstance(s, str) and len(s) >= end else "")

    def predict_regions(self, data: pd.DataFrame):
        for name in self.regions.keys():
            print(f"Predicting {name} ...")
            seqs = data[name].tolist()
            data[name] = self.process_batch(seqs)

# =========================
# Main
# =========================
def main():
    # ---- YOU MUST PROVIDE THESE ----
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # LegNet = ...  # your core model
    # UASNet = ...  # your UAS model
    #
    # Make sure LegNet and UASNet are moved to device:
    # LegNet.to(device); UASNet.to(device)

    # -------------------------------
    # Load data (IMPORTANT: confirm delimiter!)
    # If your file is comma-separated, use sep="," or just omit sep.
    # -------------------------------
    in_path = "/root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/AAA_plotting_code_and_file/cfg_promoters_evaluation/lk.csv"
    print("Loading data...")
    css_data = pd.read_csv(in_path
                                #,sep='\t'
                                )  # <- default comma CSV
    print(f"Loaded {len(css_data)} rows")

    # Sanity check columns
    if "seq" not in css_data.columns:
        raise ValueError(f"Column 'seq' not found. Columns are: {list(css_data.columns)[:20]} ...")

    # -------------------------------
    # Init processors (window fixed; step=20)
    # -------------------------------
    core_processor = CoreProcessor(LegNet, device, prom_len=500, win=80, step=20)
    uas_processor  = UASProcessor(UASNet, device, prom_len=500, win=334, step=20)

    # -------------------------------
    # Extract + Predict core windows
    # -------------------------------
    print("\n" + "=" * 60)
    print("Processing Core windows (80bp, step=20)")
    print("=" * 60)
    core_processor.extract_regions(css_data, seq_col="seq")
    core_processor.predict_regions(css_data)

    # -------------------------------
    # Extract + Predict UAS windows
    # -------------------------------
    print("\n" + "=" * 60)
    print("Processing UAS windows (334bp, step=20)")
    print("=" * 60)
    uas_processor.extract_regions(css_data, seq_col="seq")
    uas_processor.predict_regions(css_data)

    # -------------------------------
    # Aggregate stats using ACTUAL generated column names
    # -------------------------------
    cp_cols = list(core_processor.regions.keys())   # cp_0_80, cp_20_100, ..., cp_420_500
    uas_cols = list(uas_processor.regions.keys())   # uas_0_334, ..., uas_160_494

    # Ensure all exist
    missing_cp = [c for c in cp_cols if c not in css_data.columns]
    missing_uas = [c for c in uas_cols if c not in css_data.columns]
    if missing_cp or missing_uas:
        raise KeyError(f"Missing columns. cp_missing={missing_cp[:5]} ... uas_missing={missing_uas[:5]} ...")

    css_data["core_mean"] = css_data[cp_cols].mean(axis=1)
    css_data["core_max"]  = css_data[cp_cols].max(axis=1)
    css_data["uas_mean"]  = css_data[uas_cols].mean(axis=1)

    # -------------------------------
    # Save (avoid overwriting original unless you want)
    # -------------------------------
    out_path = "/root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/AAA_plotting_code_and_file/lk_cp_uas/lk.csv"
    css_data.to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")

    # -------------------------------
    # Summary
    # -------------------------------
    print("\n" + "=" * 60)
    print("Prediction Summary")
    print("=" * 60)

    print("\nCore window preds:")
    for name in cp_cols[:5]:
        print(f"  {name:12} mean={css_data[name].mean():.4f} std={css_data[name].std():.4f}")
    print(f"  ... total core windows: {len(cp_cols)}")

    print("\nUAS window preds:")
    for name in uas_cols[:5]:
        print(f"  {name:12} mean={css_data[name].mean():.4f} std={css_data[name].std():.4f}")
    print(f"  ... total UAS windows: {len(uas_cols)}")

if __name__ == "__main__":
    main()


Loading data...
Loaded 858 rows

Processing Core windows (80bp, step=20)
Predicting cp_0_80 ...


Predicting Core (80bp windows): 100%|██████████| 858/858 [00:04<00:00, 184.52it/s]


Predicting cp_20_100 ...


Predicting Core (80bp windows): 100%|██████████| 858/858 [00:04<00:00, 198.76it/s]


Predicting cp_40_120 ...


...

Predicting cp_200_280 ...


Predicting Core (80bp windows): 100%|██████████| 858/858 [00:04<00:00, 185.48it/s]


Predicting cp_220_300 ...


Predicting Core (80bp windows):   1%|          | 7/858 [00:00<00:04, 202.37it/s]


KeyboardInterrupt: 

In [3]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

def str_to_one_hot_core(seq):
    """Convert DNA sequence to one-hot encoding for Core Promoter prediction (6 channels)."""
    base_map = {'A':0, 'C':1, 'G':2, 'T':3}
    seq_tensor = torch.tensor([base_map[base] for base in seq], dtype=torch.long)
    seq_one_hot = torch.nn.functional.one_hot(seq_tensor, num_classes=4).float()
    
    singleton_channel = torch.ones((seq_one_hot.size(0), 1))  # singleton == 1
    reverse_channel = torch.zeros((seq_one_hot.size(0), 1))
    seq_one_hot = torch.cat([seq_one_hot, singleton_channel, reverse_channel], dim=1)
    return seq_one_hot

def str_to_one_hot_uas(seq):
    """Convert DNA sequence to one-hot encoding for UAS prediction (4 channels)."""
    base_map = {'A':0, 'C':1, 'G':2, 'T':3}
    seq_tensor = torch.tensor([base_map[base] for base in seq], dtype=torch.long)
    seq_one_hot = torch.nn.functional.one_hot(seq_tensor, num_classes=4).float()
    return seq_one_hot

class CoreProcessor:
    """Core Promoter prediction module."""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.regions = {
            'core_0_80': (19, 99),
            'core_80_160': (99, 179),
            'core_160_240': (179, 259),
            'core_240_320': (259, 339),
            'core_320_400': (339, 419),
            'core_400_480': (419, 499),
        }
    
    def predict_single(self, seq):
        """Predict core promoter strength from a single sequence."""
        if not seq or len(seq) == 0:
            return 0.0
            
        seq_one_hot = str_to_one_hot_core(seq).unsqueeze(0).to(self.device)  # [1, L, 6]
        seq_one_hot = seq_one_hot.permute(0, 2, 1)  # [1, 6, L] for Conv1d
        
        with torch.no_grad():
            self.model.eval()
            prediction = self.model(seq_one_hot)
            if isinstance(prediction, tuple):
                prediction = prediction[1]  # 取主输出
        return prediction.item()
    
    def process_batch(self, sequences):
        """Process a batch of core promoter sequences."""
        predictions = []
        for seq in tqdm(sequences, desc="Predicting Core Promoter Strength"):
            prediction = self.predict_single(seq)
            predictions.append(prediction)
        return predictions
    
    def extract_regions(self, data, seq_col='seq'):
        """Extract core promoter regions from sequences."""
        for region_name, (start, end) in self.regions.items():
            data[region_name] = data[seq_col].apply(
                lambda s: s[start:end] if len(s) >= end else ''
            )
    
    def predict_regions(self, data):
        """Predict all core promoter regions."""
        for region_name in self.regions.keys():
            print(f'Predicting {region_name}...')
            sequences = data[region_name].tolist()
            predictions = self.process_batch(sequences)
            data[region_name] = predictions

class UASProcessor:
    """UAS prediction module."""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.regions = {
            'uas_0_200': (0, 333),
            'uas_200_400': (166, 499),
        }
    
    def predict_single(self, seq):
        """Predict UAS strength from a single sequence."""
        if not seq or len(seq) == 0:
            return 0.0
            
        seq_one_hot = str_to_one_hot_uas(seq).unsqueeze(0).to(self.device)  # [1, L, 4]
        
        with torch.no_grad():
            self.model.eval()
            prediction = self.model(seq_one_hot)
            if isinstance(prediction, tuple):
                prediction = prediction[0]  # 取主输出
        return prediction.item()
    
    def process_batch(self, sequences):
        """Process a batch of UAS sequences."""
        predictions = []
        for seq in tqdm(sequences, desc="Predicting UAS Strength"):
            prediction = self.predict_single(seq)
            predictions.append(prediction)
        return predictions
    
    def extract_regions(self, data, seq_col='seq'):
        """Extract UAS regions from sequences."""
        for region_name, (start, end) in self.regions.items():
            data[region_name] = data[seq_col].apply(
                lambda s: s[start:end] if len(s) >= end else ''
            )
    
    def predict_regions(self, data):
        """Predict all UAS regions."""
        for region_name in self.regions.keys():
            print(f'Predicting {region_name}...')
            sequences = data[region_name].tolist()
            predictions = self.process_batch(sequences)
            data[region_name] = predictions

def main():
    """Main processing pipeline."""
    
    # 读取数据
    print("Loading data...")
    css_data = pd.read_csv(
        '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/AAA_plotting_code_and_file/cfg_promoters_evaluation/lk.csv',
        #sep='\t',
    )
    print(f"Loaded {len(css_data)} sequences")
    
    # 初始化处理器
    core_processor = CoreProcessor(LegNet, device)
    uas_processor = UASProcessor(UASNet, device)
    
    # 定义区域（在这里定义以便后续引用）
    core_regions = {
        'core_0_80': (19, 99),
        'core_80_160': (99, 179),
        'core_160_240': (179, 259),
        'core_240_320': (259, 339),
        'core_320_400': (339, 419),
        'core_400_480': (419, 499),
    }
    #core_regions = {f"cp_{i}_{i+20}": (i, i+20) for i in range(0, 500, 20)}
    
    uas_regions = {
        'uas_0_200': (0, 200),
        'uas_200_400': (200, 400),
    }
    #uas_regions = {f"uas_{i}_{i+20}": (i, i+20) for i in range(0, 400, 20)}
    
    # 处理Core Promoter区域
    print("\n" + "="*50)
    print("Processing Core Promoter regions...")
    print("="*50)
    
    core_processor.extract_regions(css_data)
    core_processor.predict_regions(css_data)
    
    # 处理UAS区域
    print("\n" + "="*50)
    print("Processing UAS regions...")
    print("="*50)
    
    uas_processor.extract_regions(css_data)
    uas_processor.predict_regions(css_data)
    
    core_cols = [
    'core_0_80', 'core_80_160', 'core_160_240',
    'core_240_320', 'core_320_400', 'core_400_480'
    ]
    uas_cols = ['uas_0_200', 'uas_200_400']
    # core_cols = [f"cp_{i}_{i+20}" for i in range(0, 500, 20)]
    # uas_cols = [f"uas_{i}_{i+20}" for i in range(0, 400, 20)]

    css_data['core_mean'] = css_data[core_cols].mean(axis=1)
    css_data['core_max'] = css_data[core_cols].max(axis=1)
    css_data['uas_mean'] = css_data[uas_cols].mean(axis=1)

    # 保存结果
    output_file_path = '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/AAA_plotting_code_and_file/lk_cp_uas/lk.csv'
    css_data.to_csv(output_file_path, index=False)
    print(f"\nCombined predictions saved to {output_file_path}")
    
    # 显示结果摘要
    print("\n" + "="*50)
    print("Prediction Summary")
    print("="*50)
    
    print("\nCore Promoter Predictions:")
    for region_name in core_regions.keys():
        mean_val = css_data[region_name].mean()
        std_val = css_data[region_name].std()
        print(f"  {region_name:15}: mean={mean_val:.4f}, std={std_val:.4f}")
    
    print("\nUAS Predictions:")
    for region_name in uas_regions.keys():
        mean_val = css_data[region_name].mean()
        std_val = css_data[region_name].std()
        print(f"  {region_name:15}: mean={mean_val:.4f}, std={std_val:.4f}")

if __name__ == "__main__":
    main()

Loading data...
Loaded 858 sequences

Processing Core Promoter regions...
Predicting core_0_80...


Predicting Core Promoter Strength:   0%|          | 0/858 [00:00<?, ?it/s]

Predicting Core Promoter Strength: 100%|██████████| 858/858 [00:04<00:00, 198.27it/s]


Predicting core_80_160...


Predicting Core Promoter Strength: 100%|██████████| 858/858 [00:04<00:00, 193.99it/s]


...


Processing UAS regions...
Predicting uas_0_200...


Predicting UAS Strength: 100%|██████████| 858/858 [00:01<00:00, 784.04it/s]


Predicting uas_200_400...


Predicting UAS Strength: 100%|██████████| 858/858 [00:00<00:00, 963.25it/s]



Combined predictions saved to /root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/AAA_plotting_code_and_file/lk_cp_uas/lk.csv

Prediction Summary

Core Promoter Predictions:
  core_0_80      : mean=10.6859, std=2.0015
  core_80_160    : mean=10.7952, std=1.9667
  core_160_240   : mean=10.9856, std=2.0617
  core_240_320   : mean=11.0607, std=2.1349
  core_320_400   : mean=11.0671, std=1.9400
  core_400_480   : mean=10.8925, std=1.4933

UAS Predictions:
  uas_0_200      : mean=-52.4880, std=46.4601
  uas_200_400    : mean=-39.5986, std=43.2110


In [1]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

def str_to_one_hot(seq):
    base_map = {'A':0, 'C':1, 'G':2, 'T':3}
    seq_tensor = torch.tensor([base_map[base] for base in seq], dtype=torch.long)
    seq_one_hot = torch.nn.functional.one_hot(seq_tensor, num_classes=4)
    
    singleton_channel = torch.ones((seq_one_hot.size(0), 1))  # singleton == 1
    reverse_channel = torch.zeros((seq_one_hot.size(0), 1))
    seq_one_hot = torch.cat([seq_one_hot, singleton_channel, reverse_channel], dim=1)
    return seq_one_hot

def predict_core_promoter(model, promoter_seq, device):
    """Predict core promoter strength from a single promoter sequence."""
    promoter_one_hot = str_to_one_hot(promoter_seq).unsqueeze(0).to(device)  # [1, L, C]
    promoter_one_hot = promoter_one_hot.permute(0, 2, 1)  # [1, C, L] —— Conv1d要求的格式
    with torch.no_grad():
        model.eval()
        predicted_strength = model(promoter_one_hot)
        if isinstance(predicted_strength, tuple):
            predicted_strength = predicted_strength[1]  # 取主输出
    return predicted_strength.item()

def process_promoters_and_infer(model, promoters, device):
    """Process a list of promoter sequences and return their predicted core promoter strength."""
    predictions = []
    for promoter in tqdm(promoters, desc="Predicting Core Promoter Strength"):
        prediction = predict_core_promoter(model, promoter, device)
        predictions.append(prediction)
    return predictions

# 读取数据
css_data = pd.read_csv(
    '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/genetic_algorithm/natural_strong_vs_synthetic_false_positive.csv',
    sep='\t'
)
#css_data = css_data[css_data['seq'].str.len() >= 500].copy()

# 截取最后500bp，并切成6段
#css_data['seq_500'] = css_data['seq'].apply(lambda s: s[-500:])

# 定义六个区段
regions = {
    'core_0_80': (19, 99),
    'core_80_160': (99, 179),
    'core_160_240': (179, 259),
    'core_240_320': (259, 339),
    'core_320_400': (339, 419),
    'core_400_480': (419, 499),
}

# 按区段提取子序列
for col_name, (start, end) in regions.items():
    css_data[col_name] = css_data['seq'].apply(lambda s: s[start:end] if len(s) >= end else '')

# 每列分别送入模型进行预测
for col_name in regions.keys():
    print(f'Predicting {col_name}...')
    promoters = css_data[col_name].tolist()
    preds = process_promoters_and_infer(LegNet, promoters, device)
    css_data[col_name] = preds  # 用预测值覆盖子序列列

# 保存结果
output_file_path = '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/SpeciesLM/genetic_algorithm/natural_strong_vs_synthetic_false_positive_subregion_prediction.csv'
css_data.to_csv(output_file_path, index=False)
print(f"Subregion predictions saved to {output_file_path}")



Predicting core_0_80...


NameError: name 'LegNet' is not defined

In [3]:
import pandas as pd
from Bio import SeqIO
import numpy as np

# Dataset
class OracleLegNetDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.shuffle = True
        self.seqs = []
        self.pa = []
        self.metadata = []  # Store metadata for later use
        self.max_len = 150
        fasta_file = '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/src/promoter_activity_109+10.fasta'
        
        for record in SeqIO.parse(fasta_file, 'fasta'):
            seq_str = str(record.seq).upper()
            truncated_seq = self.truncate_sequence(seq_str)
            self.seqs.append(truncated_seq)
            pa, spe = self.parse_seq_record(record)
            self.pa.append(pa)
            self.metadata.append((record.id, spe, pa))
        
    def __len__(self):
        return len(self.pa)

    def __getitem__(self, idx):
        seq = self.str_to_one_hot(self.seqs[idx]).clone().detach().float() # L, 4
        score = torch.tensor(self.pa[idx], dtype=torch.float)
        return seq, score, self.metadata[idx]
    
    def str_to_one_hot(self, seq):
        base_map = {'A':0, 'C':1, 'G':2, 'T':3}
        seq_tensor = torch.tensor([base_map[base] for base in seq], dtype=torch.long)
        seq_one_hot = torch.nn.functional.one_hot(seq_tensor, num_classes=4)
        
        singleton_channel = torch.ones((seq_one_hot.size(0), 1))  # singleton == 1
        reverse_channel = torch.zeros((seq_one_hot.size(0), 1))
        seq_one_hot = torch.cat([seq_one_hot, singleton_channel, reverse_channel], dim=1)
        return seq_one_hot
    
    def truncate_sequence(self, seq):
        if len(seq) > self.max_len:
            return seq[-self.max_len:]
        return seq
    
    def parse_seq_record(self, record):
        parts = record.description.split(',')
        pa = np.log(float(parts[1])+1.00001)
        spe = str(str(record.description.split(',')[0]).split(' ')[1])
        return pa, spe

# Inference
ds = OracleLegNetDataset()
dataloader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=False)

predictions = []

with torch.no_grad():
    for batch in dataloader:
        seqs, _, metadata = batch
        seqs = seqs.permute(0, 2, 1).to(device)  # Change to (batch_size, channels, length)
        #seqs = seqs.to(device)
        outputs = model(seqs)
        
        output1 = outputs[0].cpu().numpy()
        output2 = outputs[1].cpu().numpy()
        
        # 遍历 batch 中的每个样本
        for i in range(len(metadata[0])):
            gene_id = metadata[0][i]
            species = metadata[1][i]
            original_pa = metadata[2][i].item()  # 转为浮点数
            
            # 检查输出是否为标量，否则提取第一个值
            if output1[i].size == 1:  # 单个值
                predicted_pa_1 = output1[i].item()
            else:  # 数组情况，提取第一个值或根据需求自定义
                predicted_pa_1 = output1[i][0].item()
            
            if output2[i].size == 1:  # 单个值
                predicted_pa_2 = output2[i].item()
            else:  # 数组情况，提取第一个值或根据需求自定义
                predicted_pa_2 = output2[i][0].item()
            
            # 将数据记录到列表中
            predictions.append((gene_id, species, original_pa, predicted_pa_1, predicted_pa_2))
        #print(metadata)
        # for i in range(len(metadata[0])):
        #     gene_id = metadata[0][i]
        #     species = metadata[1][i]
        #     original_pa = metadata[2][i].item()  # Convert tensor to float
        #     predicted_pa = outputs[i].item()
        #     predictions.append((gene_id, species, original_pa, predicted_pa))
        #predictions.extend([(meta[0], meta[1], float(meta[2]), float(output)) for meta, output in zip(metadata, outputs)])

# Create a DataFrame and save as CSV
columns = ["Gene_ID", "Species", "Original_Promoter_Activity", "Predicted_Promoter_logprob", "Predicted_Promoter_score"]
df = pd.DataFrame(predictions, columns=columns)
output_csv_path = "/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/legnet_corepromoter_inference_output.csv"
df.to_csv(output_csv_path, index=False)

print(f"Predictions saved to {output_csv_path}")

NameError: name 'model' is not defined

In [10]:
# import pandas as pd

# df1 = pd.read_csv('/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/hybridnet_chimera_inference_output.csv')
# df2 = pd.read_csv('/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/hybridnet_corepromoter_inference_output.csv')
# df3 = pd.read_csv('/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/legnet_corepromoter_inference_output.csv')

# merged_df = pd.merge(df1, df2, on=['Gene_ID', 'Species', 'Original_Promoter_Activity'])

# merged_df['Predicted_Whole_Promoter_Activity_rank'] = merged_df['Predicted_Whole_Promoter_Activity'].rank()
# merged_df['Predicted_Core_Promoter_Activity_rank'] = merged_df['Predicted_Core_Promoter_Activity'].rank()
# merged_df['Original_Promoter_Activity_rank'] = merged_df['Original_Promoter_Activity'].rank()

# merged_df.to_csv('uas_core_promoter_predicted.csv')

import pandas as pd

# 读取 CSV 文件
df1 = pd.read_csv('/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/hybridnet_chimera_inference_output.csv')
df2 = pd.read_csv('/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/hybridnet_corepromoter_inference_output.csv')
df3 = pd.read_csv('/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/legnet_corepromoter_inference_output.csv')

# 合并 df1 和 df2
merged_df = pd.merge(df1, df2, on=['Gene_ID', 'Species', 'Original_Promoter_Activity'])

# 合并 df3
final_merged_df = pd.merge(merged_df, df3, on=['Gene_ID', 'Species', 'Original_Promoter_Activity'])

# 添加排名列
final_merged_df['Predicted_Whole_Promoter_Activity_rank'] = final_merged_df['Predicted_Whole_Promoter_Activity'].rank()
final_merged_df['Predicted_Core_Promoter_Activity_rank'] = final_merged_df['Predicted_Core_Promoter_Activity'].rank()
final_merged_df['Original_Promoter_Activity_rank'] = final_merged_df['Original_Promoter_Activity'].rank()
final_merged_df['Predicted_LegNet_Activity_rank'] = final_merged_df['Predicted_Promoter_score'].rank()  # 假设 df3 中的活动列为 Predicted_Promoter_Activity_1

# 保存到新的 CSV 文件
output_path = '/root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/uas_core_promoter_predicted_with_legnet.csv'
final_merged_df.to_csv(output_path, index=False)

print(f"Final merged CSV saved to {output_path}")


Final merged CSV saved to /root/autodl-tmp/graph_model/Diffusion_directed_evolution/seqs_and_embeddings/uas_core_promoter_predicted_with_legnet.csv
